# PER-CLASS × NORM — M1a vs M0 (10k, frozen 12-AA `9162ce44`)

**CPU runtime** (Runtime → Change runtime type → **CPU**) — do not spend GPU quota. Whole notebook runs in **< 5 min**.

Reads the **existing** 10k mask sidecars (same masks used for the paired bootstrap + concordance) and reports, per class (n=1000):
union accuracy (AND over 12 components), per-norm robust (AND over that norm's 4 attacks), Δ = M1a − M0, per-class paired bootstrap CI, sign test across the 10 classes, and bottom-3 lift.

Outputs write **straight to Drive**: `union_bench/perclass/{perclass_m1a_m0.csv, perclass_delta.md}`. **Skip-if-done**: if both exist, the notebook just re-prints them.

> **No clean column** — the mask sidecar stores only the 12 attack masks (no clean mask), and we never run new inference.

## Cell 1 — Drive mount + config (Drive-direct)

In [ ]:
from google.colab import drive; drive.mount('/content/drive')

import os
DRIVE       = '/content/drive/MyDrive/attackdro'
RESULTS_DIR = f'{DRIVE}/union_bench'                 # canonical, Drive-direct
OUTDIR      = f'{RESULTS_DIR}/perclass'
CSV_OUT     = f'{OUTDIR}/perclass_m1a_m0.csv'
MD_OUT      = f'{OUTDIR}/perclass_delta.md'
CODE_ZIP    = f'{DRIVE}/attackdro_code.zip'          # M1a masks ship inside the zip
CIFAR_TARGZ = f'{DRIVE}/cifar-10-python.tar.gz'      # reuse upload if present (else torchvision downloads)

# 10k mask candidates, in priority order (union_bench canonical -> legacy audit_out -> zip for M1a)
CAND = {
 'M1a': [f'{RESULTS_DIR}/M1a/10k/masks_multinorm_v1.npz', f'{DRIVE}/audit_out/M1a/10k/masks_multinorm_v1.npz'],
 'M0' : [f'{RESULTS_DIR}/M0/10k/masks_multinorm_v1.npz',  f'{DRIVE}/audit_out/M0/10k/masks_multinorm_v1.npz'],
}
os.makedirs(OUTDIR, exist_ok=True)
DONE = os.path.exists(CSV_OUT) and os.path.exists(MD_OUT)
print('RESULTS_DIR =', RESULTS_DIR)
print('outputs     =', CSV_OUT, '|', MD_OUT)
print('SKIP-IF-DONE:', DONE)

## Cell 2 — Resolve mask paths + introspect schema (stop & print if unexpected, no guessing)

In [ ]:
import numpy as np, json, zipfile, os

def resolve(arm):
    for p in CAND[arm]:
        if os.path.exists(p): return p
    # M1a masks are baked into attackdro_code.zip -> extract into union_bench/ once
    if arm == 'M1a' and os.path.exists(CODE_ZIP):
        member = 'results/eval/union_bench/M1a/10k/masks_multinorm_v1.npz'
        with zipfile.ZipFile(CODE_ZIP) as z:
            if member in z.namelist():
                dst = CAND['M1a'][0]; os.makedirs(os.path.dirname(dst), exist_ok=True)
                with z.open(member) as src, open(dst, 'wb') as f: f.write(src.read())
                print('extracted M1a masks from code zip ->', dst); return dst
    return None

PATHS = {a: resolve(a) for a in ['M1a','M0']}
for a,p in PATHS.items(): print(f'{a}: {p}')
missing = [a for a,p in PATHS.items() if p is None]
assert not missing, (f"masks not found for {missing}. Expected one of {[CAND[a] for a in missing]} "
                     f"(M0 was audited on Colab -> check audit_out/M0/10k/, or run the audit_gate merge cell).")

MASKS, META = {}, {}
for a,p in PATHS.items():
    z = np.load(p)
    MASKS[a] = {k: z[k] for k in z.files if k != 'metadata_json'}
    if 'metadata_json' in z.files:
        raw = z['metadata_json']; META[a] = json.loads(raw.item() if raw.shape==() else str(raw))
    print(f'\n=== {a} :: {p}')
    print('  keys:', list(z.files))
    for k,v in list(MASKS[a].items())[:3]:
        print(f'   {k}: shape={v.shape} dtype={v.dtype} head={v[:8].astype(int).tolist()} mean={v.astype(bool).mean():.4f}')
    print(f'   ... ({len(MASKS[a])} component masks total)')
    if a in META:
        m=META[a]
        print('  meta: n_examples=%s  convention=%s  subset_sha=%s' % (
              m.get('n_examples'), m.get('mask_convention'), str(m.get('audit_subset_sha256'))[:16]))
        print('        checkpoint=%s' % m.get('checkpoint_path'))

# ---- schema gate: refuse to guess ----
ok = True
for a in PATHS:
    m = MASKS[a]
    if len(m) != 12: print(f'!! {a}: expected 12 component masks, got {len(m)}'); ok=False
    for k,v in m.items():
        if v.shape != (10000,): print(f'!! {a}.{k}: expected shape (10000,), got {v.shape}'); ok=False; break
        if v.dtype != np.bool_: print(f'!! {a}.{k}: expected bool, got {v.dtype}'); ok=False; break
    conv = META.get(a,{}).get('mask_convention')
    if conv not in (None,'true_means_robust_false_means_failed'):
        print(f'!! {a}: unexpected mask_convention={conv}'); ok=False
if META.get('M1a',{}).get('audit_subset_sha256') != META.get('M0',{}).get('audit_subset_sha256'):
    print('!! arms audited on DIFFERENT subsets — refusing to pair'); ok=False
assert ok, 'SCHEMA MISMATCH — schema printed above. Stopping (no guessing).'
print('\nschema OK: 2 arms x 12 bool masks (10000,), convention=true_means_robust, same subset.')

## Cell 3 — CIFAR-10 test labels + verify index alignment (labels only, no GPU)

In [ ]:
import tarfile, numpy as np, os
ROOT_DATA='/content/data'; os.makedirs(ROOT_DATA, exist_ok=True)
if not os.path.exists(f'{ROOT_DATA}/cifar-10-batches-py/test_batch') and os.path.exists(CIFAR_TARGZ):
    with tarfile.open(CIFAR_TARGZ) as t: t.extractall(ROOT_DATA)          # reuse the Drive upload
import torchvision
ds = torchvision.datasets.CIFAR10(root=ROOT_DATA, train=False,
                                  download=not os.path.exists(f'{ROOT_DATA}/cifar-10-batches-py/test_batch'))
y_full = np.array(ds.targets); CLASSES = ds.classes
print('CIFAR-10 test labels:', y_full.shape, '| per-class:', np.bincount(y_full))

# align mask position -> test index via the audit subset the masks name
sub_rel = META['M1a'].get('audit_subset_path','results/audit/subsets/cifar10_test_10000_full_v3A.json')
idx = None
if os.path.exists(CODE_ZIP):
    import zipfile
    with zipfile.ZipFile(CODE_ZIP) as z:
        if sub_rel in z.namelist(): idx = json.loads(z.read(sub_rel))['indices']
if idx is None:
    idx = list(range(10000)); print('subset json not in zip -> assuming identity order (verified below)')
Y = y_full[np.array(idx)]
assert len(idx)==10000, f'subset n={len(idx)} != 10000'
assert (np.bincount(Y)==1000).all(), f'class counts not 1000/class: {np.bincount(Y)}'
print('subset:', sub_rel)
print('identity order:', idx==list(range(10000)), '| aligned labels:', Y.shape, '| per-class:', np.bincount(Y))

## Cell 4 — Compute per-class union + per-norm (AND rules)

In [ ]:
NORMS=['linf','l2','l1']
def union_all(m):
    u=np.ones(10000,bool)
    for v in m.values(): u &= v.astype(bool)
    return u
def per_norm(m,norm):
    ks=[k for k in m if k.endswith(norm)]        # 'linf' never matches '...l1' (suffix-safe)
    assert len(ks)==4, f'{norm}: expected 4 attacks, got {ks}'
    u=np.ones(10000,bool)
    for k in ks: u &= m[k].astype(bool)
    return u

U   = {a: union_all(MASKS[a]) for a in ['M1a','M0']}
PN  = {a: {nm: per_norm(MASKS[a],nm) for nm in NORMS} for a in ['M1a','M0']}
for a in ['M1a','M0']:
    print(f"{a}: union={U[a].mean():.4f}  " + "  ".join(f"{nm}={PN[a][nm].mean():.4f}" for nm in NORMS))
print(f"overall Δ union = {U['M1a'].mean()-U['M0'].mean():+.4f}")

## Cell 5 — Stats: per-class paired bootstrap (B=2000, seed=0) · sign test · bottom-3 lift

In [ ]:
from math import comb
def boot_ci(a,b,B=2000,seed=0):
    rng=np.random.default_rng(seed); n=len(a); d=np.empty(B)
    for i in range(B):
        j=rng.integers(0,n,n); d[i]=a[j].mean()-b[j].mean()
    return float(a.mean()-b.mean()), float(np.quantile(d,0.025)), float(np.quantile(d,0.975))
def sign_test(deltas):
    pos=sum(1 for d in deltas if d>0); neg=sum(1 for d in deltas if d<0); n=pos+neg
    if n==0: return pos,neg,1.0
    cdf=lambda k: sum(comb(n,i) for i in range(0,k+1))/2**n
    p=2*min(cdf(pos), 1-cdf(pos-1) if pos>0 else 1.0)
    return pos,neg,float(min(1.0,p))

PC={}
for c in range(10):
    s=(Y==c); rec={'n':int(s.sum())}
    rec['M0_union']=float(U['M0'][s].mean()); rec['M1a_union']=float(U['M1a'][s].mean())
    d,lo,hi=boot_ci(U['M1a'][s],U['M0'][s]); rec.update(delta_union=d, ci_lo=lo, ci_hi=hi)
    for nm in NORMS:
        rec[f'M1a_{nm}']=float(PN['M1a'][nm][s].mean()); rec[f'M0_{nm}']=float(PN['M0'][nm][s].mean())
        rec[f'delta_{nm}']=rec[f'M1a_{nm}']-rec[f'M0_{nm}']
    PC[CLASSES[c]]=rec

dU=[PC[c]['delta_union'] for c in PC]
pos,neg,p_union=sign_test(dU)
sgn={nm: sign_test([PC[c][f'delta_{nm}'] for c in PC]) for nm in NORMS}
weakest=sorted(PC, key=lambda c: PC[c]['M0_union'])[:3]
bottom3=float(np.mean([PC[c]['delta_union'] for c in weakest]))
cmax=max(PC,key=lambda c:PC[c]['delta_union']); cmin=min(PC,key=lambda c:PC[c]['delta_union'])
print('stats done — sign test union: %d+/%d- p=%.4f | bottom-3 lift %+.4f' % (pos,neg,p_union,bottom3))

## Cell 6 — Write to Drive + print (skip-if-done just re-prints)

In [ ]:
import csv, os
HDR = "| class | M0 union | M1a union | Δ (95% CI) | Δℓ∞ | Δℓ₂ | Δℓ₁ |\n|---|---|---|---|---|---|---|"
rows_md=[f"| {c:10s} | {PC[c]['M0_union']:.3f} | {PC[c]['M1a_union']:.3f} | "
         f"{PC[c]['delta_union']:+.3f} [{PC[c]['ci_lo']:+.3f},{PC[c]['ci_hi']:+.3f}] | "
         f"{PC[c]['delta_linf']:+.3f} | {PC[c]['delta_l2']:+.3f} | {PC[c]['delta_l1']:+.3f} |" for c in PC]
SUMMARY=[
 f"- **Sign test (union, n=10 classes):** {pos} positive / {neg} negative → exact two-sided **p = {p_union:.4f}**",
 "- **Sign test per norm:** " + " · ".join(
    f"ℓ{'∞' if nm=='linf' else nm[-1]} {sgn[nm][0]}+/{sgn[nm][1]}− (p={sgn[nm][2]:.4f})" for nm in NORMS),
 f"- **Bottom-3 lift** (3 classes where M0 is weakest on union: {', '.join(weakest)}): **Δ = {bottom3:+.4f}**",
 f"- **Max Δ:** {cmax} ({PC[cmax]['delta_union']:+.4f})  ·  **Min Δ:** {cmin} ({PC[cmin]['delta_union']:+.4f})",
]
if DONE:
    print('SKIP-IF-DONE — outputs already exist; re-printing:\n'); print(open(MD_OUT).read())
else:
    with open(CSV_OUT,'w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=['class','arm','union']+NORMS); w.writeheader()
        for c in PC:
            for arm in ['M1a','M0']:
                w.writerow({'class':c,'arm':arm,'union':PC[c][f'{arm}_union'],
                            **{nm:PC[c][f'{arm}_{nm}'] for nm in NORMS}})
    body=("# Per-class × norm — M1a vs M0 (10k, frozen 12-AA 9162ce44)\n\n"
          "Union = per-example AND over 12 components; per-norm = AND over that norm's 4 attacks. "
          "Per-class paired bootstrap B=2000, seed=0, resampled within class (n=1000/class). "
          "No clean column (sidecar stores no clean mask; no new inference).\n\n"
          + HDR + "\n" + "\n".join(rows_md) + "\n\n" + "\n".join(SUMMARY) + "\n")
    open(MD_OUT,'w').write(body)
    print('saved ->', CSV_OUT); print('saved ->', MD_OUT); print()
    print(HDR); print("\n".join(rows_md)); print(); print("\n".join(SUMMARY))